# Resume job-category model — evaluation

This notebook evaluates the **Naive Bayes** resume classifier used in this project (`resume/ml/`). It reports **accuracy**, **per-class precision / recall / F1**, **macro** and **weighted** averages, and a **confusion matrix**.

Run all cells from the project root, or keep the path logic in the first code cell so `resume` imports resolve.

In [ ]:
# Optional: install evaluation stack (skip if already installed)
%pip install -q scikit-learn matplotlib pandas

: 

In [ ]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)

# Resolve Django project root (directory that contains manage.py)
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "manage.py").exists() and (p / "resume").is_dir():
            return p
    raise FileNotFoundError(
        "Could not find project root (manage.py). Open the notebook from the repo or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from resume.ml.naive_bayes import NaiveBayesClassifier
from resume.ml.train_model import (
    evaluate_model,
    _load_dataset,
    _rebalance_training_rows,
    _select_dataset_path,
    _to_feature_text,
)

print("Project root:", PROJECT_ROOT)

## Load dataset and split (same defaults as `train_model.train_and_save_model`)

Adjust `SEED`, `TRAIN_RATIO`, and rebalance flags to match how you train in production.

In [ ]:
SEED = 42
TRAIN_RATIO = 0.8
REBALANCE_CLASSES = True

dataset_path = PROJECT_ROOT / "datasets" / "structured_resume_dataset.csv"
cleaned_path = PROJECT_ROOT / "datasets" / "structured_resume_dataset.cleaned.csv"

selected = _select_dataset_path(
    str(dataset_path),
    str(cleaned_path),
    prefer_cleaned_dataset=True,
)
rows = _load_dataset(selected)
if not rows:
    raise ValueError(f"No rows in dataset: {selected}")

random.Random(SEED).shuffle(rows)
split_idx = int(len(rows) * TRAIN_RATIO)
train_rows, test_rows = rows[:split_idx], rows[split_idx:]

if REBALANCE_CLASSES:
    train_rows = _rebalance_training_rows(train_rows, seed=SEED)

X_train = [_to_feature_text(r) for r in train_rows]
y_train = [r["job_category"] for r in train_rows]
X_test = [_to_feature_text(r) for r in test_rows]
y_test = [r["job_category"] for r in test_rows]

print("Dataset file:", selected)
print("Train rows:", len(train_rows), "| Test rows:", len(test_rows))
print("Labels (sorted):", sorted(set(y_train) | set(y_test)))

## Train classifier and predict on the hold-out set

In [ ]:
clf = NaiveBayesClassifier()
clf.train(X_train, y_train)
raw_preds = clf.predict(X_test)
y_pred = [p["label"] for p in raw_preds]

# Example: single-row confidence (same as API predictor)
sample = raw_preds[0]
print("Sample prediction:", sample)

## Core metrics

- **Project `evaluate_model`**: same dictionary written to `media/models/model_metrics.json` during training (per-class precision/recall/F1 plus accuracy).
- **scikit-learn**: macro / weighted averages and a printable classification report.

In [ ]:
labels = sorted(set(y_test) | set(y_pred))

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy (sklearn): {acc:.4f}\n")

prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=labels, average="macro", zero_division=0
)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=labels, average="weighted", zero_division=0
)
summary = pd.DataFrame(
    {
        "average": ["macro", "weighted"],
        "precision": [prec_m, prec_w],
        "recall": [rec_m, rec_w],
        "f1": [f1_m, f1_w],
    }
)
print("Macro / weighted averages:")
display(summary)

print("\nClassification report (per class):")
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))

project_metrics = evaluate_model(y_test, y_pred)
print("\nProject evaluate_model() (JSON-serializable dict):")
print(json.dumps(project_metrics, indent=2))

## Tabular per-class metrics (from project helper)

Keys like `{label}_precision` match `train_model.evaluate_model`.

In [ ]:
per_class_rows = []
for cat in labels:
    per_class_rows.append(
        {
            "category": cat,
            "precision": project_metrics.get(f"{cat}_precision", 0.0),
            "recall": project_metrics.get(f"{cat}_recall", 0.0),
            "f1": project_metrics.get(f"{cat}_f1", 0.0),
        }
    )
pd.DataFrame(per_class_rows).sort_values("f1", ascending=False).reset_index(drop=True)

## Confusion matrix

Rows = true label, columns = predicted label.

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.6), max(6, len(labels) * 0.5)))
disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=labels,
    ax=ax,
    colorbar=True,
    xticks_rotation=45,
)
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## Optional: metrics saved with the last trained artifact

If you have already run `python -m resume.ml.train_model` (or the app auto-trained), compare the JSON on disk with this notebook’s split (they match only if you use the same `SEED`, paths, and rebalance settings).

In [ ]:
metrics_json = PROJECT_ROOT / "media" / "models" / "model_metrics.json"
if metrics_json.exists():
    with open(metrics_json, "r", encoding="utf-8") as f:
        on_disk = json.load(f)
    print("On-disk model_metrics.json:")
    print(json.dumps(on_disk, indent=2))
else:
    print("No file at", metrics_json)